## Phase 1: Raw Data Consolidation

I'm loading all four workbooks (Electronics, Home Appliances, Fashion, Books). Each has 
multiple category sheets except Books, which is a single sheet. I'm tagging every row with 
`source_sheet` (the original sheet name) and `workbook` before concatenation, so this 
ground-truth signal survives consolidation. I'll use it later as a cross-check against `category_leaf`/`category_normalized`.

In [3]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")

workbooks = {
    "Electronics": "Electronics.xlsx",
    "Home Appliances": "Home Appliances.xlsx",
    "Fashion": "Fashion.xlsx",
    "Books": "Books.xlsx",
}

all_sheets = []
sheet_shapes = {}

for workbook_name, filename in workbooks.items():
    sheets = pd.read_excel(DATA_DIR / filename, sheet_name=None, na_values=["—"])
    for sheet_name, df in sheets.items():
        df = df.copy()
        df["source_sheet"] = sheet_name
        df["workbook"] = workbook_name
        sheet_shapes[f"{workbook_name} / {sheet_name}"] = df.shape
        all_sheets.append(df)

raw_combined = pd.concat(all_sheets, ignore_index=True, sort=False)
raw_combined.shape

(8287, 26)

### Sanity checks: row counts and column consistency across sheets

Before doing anything else, I want to confirm total row volume against my ~5-8K target, and 
check whether any sheet is missing columns the others have. A missing column would just show 
up as extra NaNs post-concat and could go unnoticed, so I'm checking it explicitly here.

In [ ]:
# Row count per sheet 

shape_df = pd.DataFrame(sheet_shapes, index=["rows","cols"]).T.sort_values("rows", ascending=False)
print(shape_df)
print(f"\nTotal rows combined: {raw_combined.shape[0]}")
print(f"\nTotal columns after concat: {raw_combined.shape[1]}")

                                     rows  cols
Books / Books                        1000    26
Electronics / Toys                    600    26
Home Appliances / Fridge              600    26
Fashion / Sunglass                    600    26
Home Appliances / Kitchen Appliance   560    26
Fashion / Shirt                       559    26
Fashion / Shoes                       520    26
Electronics / Keyboard                511    26
Electronics / Laptops                 498    26
Fashion / Cloth                       480    26
Fashion / Handbag                     480    26
Home Appliances / AC                  480    26
Electronics / Feature Phone           440    26
Home Appliances / TV                  399    26
Electronics / Softwares               360    26
Electronics / HDD                     200    26

Total rows combined: 8287

Total columns after concat: 26


8,287 rows total across 16 sheets — within my 5-8K target. HDD is the smallest at 200 rows.

I stopped scraping that category early because past ~200 the category page started surfacing 
off-topic products, so continuing would've added contamination rather than real coverage.

In [5]:
# Column consistency check — which sheets are missing which original columns
original_cols_per_sheet = {}
for workbook_name, filename in workbooks.items():
    sheets = pd.read_excel(DATA_DIR / filename, sheet_name=None, nrows=0)
    for sheet_name, df in sheets.items():
        original_cols_per_sheet[f"{workbook_name} / {sheet_name}"] = set(df.columns)

all_cols = set.union(*original_cols_per_sheet.values())
for sheet_id, cols in original_cols_per_sheet.items():
    missing = all_cols - cols
    if missing:
        print(f"{sheet_id} is missing: {missing}")

Since there is no output, every sheet has the same 26 columns after tagging. So I'm working with a clean union with no missing-column gaps to patch before moving on.

### Inspecting the Category column

Before I can build the normalization mapping, I need to see every unique raw `Category` 
value and how often each occurs. This is also where I'll spot the overlapping-path problem 
mentioned in my planning notes — e.g. "Software" potentially showing up under two different 
top-level categories.

In [14]:
category_counts = raw_combined["Category"].value_counts()
category_counts

Category
Media, Music & Books > Books > English Books                                           841
Fashion > Men > Clothing                                                               555
Fashion > Men > Shoes                                                                  495
Fashion > Women > Clothing                                                             478
Bags and Travel > Women Bags                                                           429
                                                                                      ... 
TV, Audio / Video, Gaming & Wearables > Console Gaming > Xbox                            1
TV, Audio / Video, Gaming & Wearables > Audio > Home Entertainment > Component S...      1
Watches Sunglasses Jewellery > Eyewear > Women                                           1
Bags and Travel                                                                          1
Media, Music & Books > Books                                                     

In [ ]:
# Breadcrumb depth check — confirms the inconsistent-depth issue and shows its spread

print(f"Unique raw Category values: {category_counts.shape[0]}")

depth = raw_combined["Category"].str.split(">").str.len()
depth.value_counts().sort_index()


Unique raw Category values: 164


Category
1.0     431
2.0    2897
3.0    4502
4.0     450
Name: count, dtype: int64

164 unique raw Category values, with depth ranging from 1 to 4 segments. Depth 1 rows have 
no ">" separator at all, so I need to handle that as a valid case, not an error, when I build 
category_l1 and category_leaf. I'm exporting the full list to a CSV so I can review it outside 
the notebook and decide on normalization mappings for Section 6.

In [16]:
category_counts.to_csv("data/category_value_counts.csv")